# ⏳ Barras de Progresso com tqdm

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Pandas](https://img.shields.io/badge/Pandas-150458?style=flat-square&logo=pandas&logoColor=white)
![tqdm](https://img.shields.io/badge/tqdm-FFC107?style=flat-square)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-tqdm%20%7C%20pandas%20%7C%20for-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Iniciante-brightgreen)
![Pré-req](https://img.shields.io/badge/pré--req-dataframe%20%7C%20for%20%7C%20merge-purple)
![Biblioteca](https://img.shields.io/badge/requer-pip%20install%20pandas%20&&_tqdm-orange)

> E quando o nosso código demora muito? Será que travou? Quanto tempo vai demorar? Em loops grandes (centenas de milhares de linhas), rodar "no escuro" é arriscado — o `tqdm` transforma qualquer iterável em uma barra de progresso, com % concluído, tempo decorrido e estimativa de tempo restante.

## 📋 Conteúdo

1. [Introdução: módulo tqdm](#-1-introdução-módulo-tqdm)
2. [Preparando a base](#-2-preparando-a-base)
3. [Desafio](#-3-desafio)
4. [Solução com for e tqdm](#-4-solução-com-for-e-tqdm)

## ⏳ 1. Introdução: módulo tqdm

O `tqdm` (do árabe 🇸🇦 *taqaddum*, "progresso") envolve qualquer iterável e passa a mostrar uma barra de progresso automaticamente, sem mudar a lógica do loop.

```python
from tqdm import tqdm

for item in tqdm(iteravel, total=tamanho_total):
    ...
```

| Parâmetro | O que faz |
|-----------|-----------|
| `iteravel` | Qualquer coisa percorrível em um for — lista, range, `.iterrows()` de um DataFrame, etc. |
| `total` | Quantidade total de itens, usada para calcular % concluído e o tempo estimado (ETA). Sem isso, o tqdm só conta, sem saber quanto falta. |

<h2 align="left">🐼 <img src="https://img.shields.io/badge/Pandas-111827?style=flat-square&logo=pandas&logoColor=white"/> 2. Preparando a base</h2>

- Vamos importar os arquivos csv da Empresa Contoso e tratar como fizemos ao longo desse módulo

In [ ]:
import pandas as pd
from cores import *
#importando os arquivos das subpastas

vendas_df = pd.read_csv(r'../spec/Contoso - Vendas - 2017.csv', sep=';', encoding='latin1')
produtos_df = pd.read_csv(r'../spec/Contoso - Produtos Cadastro Completo.csv', sep=';', encoding='latin1')
lojas_df = pd.read_csv(r'../spec/Contoso - Lojas.csv', sep=';', encoding='latin1')
clientes_df = pd.read_csv(r'../spec/Contoso - Clientes.csv', sep=';', encoding='latin1')

#a 1ª coluna de alguns arquivos vem com um caractere de encoding grudado no nome (ex: 'ÿID Loja'); corrigindo pelo nome real

produtos_df = produtos_df.rename(columns={produtos_df.columns[0]: 'Nome do Produto'})
lojas_df = lojas_df.rename(columns={lojas_df.columns[0]: 'ID Loja'})
clientes_df = clientes_df.rename(columns={clientes_df.columns[0]: 'ID Cliente'})

#limpando apenas as colunas que queremos

clientes_df = clientes_df[['ID Cliente', 'E-mail']]
produtos_df = produtos_df[['ID Produto', 'Nome do Produto']]
lojas_df = lojas_df[['ID Loja', 'Nome da Loja']]

#mesclando e renomeando os dataframes

vendas_df = vendas_df.merge(produtos_df, on='ID Produto')
vendas_df = vendas_df.merge(lojas_df, on='ID Loja')
vendas_df = vendas_df.merge(clientes_df, on='ID Cliente').rename(columns={'E-mail': 'E-mail do Cliente'})

display(vendas_df[0:21])

## 🎯 3. Desafio

Imagine que a Loja Contoso Roma (`ID 222`), para tentar burlar o sistema de metas, diminuiu 1 da quantidade devolvida de todas as vendas que ela teve. Descobrindo isso, precisamos ajeitar a base.

Regras:
- Toda venda da Loja Contoso Roma (`ID Loja == 222`) precisa somar 1 de volta na `Quantidade Devolvida`.
- As demais lojas não são afetadas.

Farei com um for, principalmente por motivos didáticos, mas teríamos outras formas de fazer isso também (por exemplo, `.loc[]` vetorizado — sem for, sem tqdm).

## ⏳🔁 4. Solução com for e tqdm

Com quase 1 milhão de linhas em `vendas_df`, um `for` percorrendo linha a linha demora — exatamente o cenário em que uma barra de progresso ajuda a saber se o código travou ou só está processando.

```python
for indice, linha in tqdm(dataframe.iterrows(), total=dataframe.shape[0]):
    ...
```

| Parâmetro 🔑 | O que faz 🔓 |
|-----------|-----------|
| `dataframe.iterrows()` | Percorre o DataFrame linha a linha, retornando `(índice, linha)` a cada passo |
| `total=dataframe.shape[0]` | Número de linhas do DataFrame — é o que permite a barra calcular o "100%" |

In [ ]:
from tqdm import tqdm

for indice_linha, linha in tqdm(vendas_df.iterrows(), total=vendas_df.shape[0]):
    if (linha['ID Loja'] == 222):
        vendas_df.loc[indice_linha, 'Quantidade Devolvida'] += 1

print(f"{CinzaClaro} Processamento{Reset} {VerdeClaro}concluído! {Reset}")
display(vendas_df[0:21])

> ⚠️ **Cuidado:** `iterrows()` é ótimo para aprender, mas não é a forma mais rápida — para DataFrames muito grandes, uma solução vetorizada (`vendas_df.loc[vendas_df['ID Loja'] == 222, 'Quantidade Devolvida'] += 1`) resolve o mesmo problema em uma fração do tempo, sem loop.